# ChopDock — Step-by-Step Walkthrough

This notebook walks through the full docking pipeline: from a SMILES string and a protein PDB to a docked 3D pose. Each step is visualised so you can see exactly what's happening geometrically, not just numerically.

The approach is *shape-based*: we voxelise the binding pocket, represent it as a cloud of dummy atoms, and optimise how well the candidate molecule fits inside that cloud. No force fields during the search — just shape overlap.

In [ ]:
import sys, os

if 'google.colab' in sys.modules:
    %pip install rdkit scipy py3Dmol -q
    if not os.path.exists('chopdock'):
        !git clone https://github.com/JanoschMenke/chopdock.git
    if os.path.basename(os.getcwd()) != 'chopdock':
        os.chdir('chopdock')

In [ ]:
import numpy as np
import py3Dmol
from io import StringIO
from rdkit import Chem
from rdkit.Chem import SDWriter
from scipy.spatial.transform import Rotation

from src.config import Config
from src.pocket import build_pocket_grid, pocket_to_pseudomol
from src.io import load_reference_ligand
from src.conformers import embed_candidate
from src.alignment import (
    align_to_reference, score_pose,
    get_coords, set_coords,
    dock_candidate, mmff_energy,
)
from src.refinement import refine_pose, minimize_pose


def mol_to_sdf(mol, conf_id=-1):
    buf = StringIO()
    w = SDWriter(buf)
    w.write(mol, confId=conf_id)
    w.close()
    return buf.getvalue()

# ── Visual style constants ────────────────────────────────────────────────
COL_PROTEIN   = "#FFFFFF"   # neutral gray
COL_REFERENCE = "#E7B510"   # orange
COL_CANDIDATE = '#2980B9'   # blue
COL_POCKET    = 'cyan'

def bs(color, sr=0.35, tr=0.15):
    """Ball-and-stick style dict for a fixed colour."""
    return {'stick': {'color': color, 'radius': tr},
            'sphere': {'color': color, 'radius': sr}}


## Inputs

Set your protein, reference ligand, and candidate SMILES here before running the rest of the notebook.

In [ ]:
cfg = Config()

cfg.protein_pdb      = 'data/example_data/7XKJ_protein.pdb'
cfg.reference_ligand = 'data/example_data/7XKJ_ligand.sdf'
cfg.candidate_smiles = 'CC1C=CC(NC(C2C=CC(C3C=CC=C3)=CC=2)=O)=CC=1NC1N=C(C2C=CC=NC=2)C=CN=1'

## Step 1 — Loading Structures

We need two inputs:
- **Protein PDB**: the receptor we're docking into
- **Reference ligand SDF**: a known binder in its crystallographic pose — this defines *where* the pocket is

The reference ligand is only used to locate the pocket and provide a shape template for alignment. We never assume our candidate is chemically similar to it. Although that helps of course.

In [ ]:
protein = Chem.MolFromPDBFile(cfg.protein_pdb, sanitize=False, removeHs=False, proximityBonding=False)
reference = load_reference_ligand(cfg.reference_ligand)

print(f"Protein:           {protein.GetNumAtoms()} atoms")
print(f"Reference ligand:  {reference.GetNumAtoms()} atoms")

In [ ]:
view = py3Dmol.view(width=800, height=500)
with open(cfg.protein_pdb) as f:
    view.addModel(f.read(), 'pdb')
view.setStyle({}, {'cartoon': {'color': COL_PROTEIN, 'opacity': 0.9}})
view.addModel(mol_to_sdf(reference), 'sdf')
view.setStyle({'model': 1}, bs(COL_REFERENCE))
view.zoomTo({'model': 1})
view.show()

## Step 2 — Building the Pocket Grid

We carve a bounding box around the reference ligand, lay down a 3D voxel grid, and mark every voxel that's within vdW+probe distance of a protein atom as **occupied**. The remaining voxels are the empty pocket.

Key parameters:

| Parameter | Default | Effect |
|-----------|---------|--------|
| `padding` | 4.0 Å | How far beyond the ligand extent to extend the box. Increase if you want to capture a slightly larger pocket. |
| `spacing` | 0.5 Å | Voxel size. Finer = more accurate pocket shape, but memory grows as O(spacing⁻³). |
| `probe_radius` | 1.4 Å | Inflates each protein atom by this much before marking voxels occupied. Larger values make the pocket tighter. |

In [ ]:
origin, spacing, empty, ref_centroid = build_pocket_grid(
    protein, reference,
    padding=cfg.padding,
    spacing=cfg.spacing,
    probe_radius=cfg.probe_radius,
    vdw_radii=cfg.vdw_radii,
)
print(f"Grid dimensions:  {empty.shape}")
print(f"Total voxels:     {empty.size:,}")
print(f"Empty (pocket):   {empty.sum():,}  ({100*empty.mean():.1f}%)")
print(f"Occupied:         {(~empty).sum():,}  ({100*(~empty).mean():.1f}%)")

## Step 3 — Pocket Representation: `surface_only` and `subsample`

We convert the empty-voxel grid into an RDKit pseudo-molecule (a cloud of bondless dummy atoms). This is what the shape scorer sees. Two parameters control how detailed that cloud is:

**`surface_only`** if `True`, we erode the pocket inward by one voxel and keep only the shell. The interior is discarded it can lead to slightlz less results.

**`subsample`** keep every Nth voxel in each dimension. `subsample=2` retains 1/8th of the surface points. Lower values = denser cloud = more accurate shape representation, but `score_pose` is called ~2000 times during docking so every extra atom costs time.

The **reference ligand score** (ProtrudeDist of the crystallographic pose against the pocket) is practical lower bound, technicallz zou can always score lower but it gives you an indication of how good the pseudo mol captures the actual pocket and how to interpret score for zour candidate molecule. 

In [ ]:
settings = [
    ("Full interior  (surface_only=False, subsample=3)", False, 3),
    ("Surface only   (surface_only=True,  subsample=1)", True,  1),
    ("Default        (surface_only=True,  subsample=2)", True,  2),
]

results = {}
for label, surf, sub in settings:
    pseudo_tmp, _ = pocket_to_pseudomol(
        empty, origin, spacing, ref_centroid,
        surface_only=surf, subsample=sub,
    )
    ref_score = score_pose(reference, 0, pseudo_tmp)
    results[label] = (pseudo_tmp, ref_score)
    print(f"{label}")
    print(f"  atoms: {pseudo_tmp.GetNumAtoms():,}   ref score: {ref_score:.4f}\n")

In [ ]:
pseudo_interior, _ = pocket_to_pseudomol(
    empty, origin, spacing, ref_centroid, surface_only=False, subsample=2
)
pseudo_surface, _ = pocket_to_pseudomol(
    empty, origin, spacing, ref_centroid, surface_only=True, subsample=2
)

def _pocket_view(pseudo, color, opacity, title):
    view = py3Dmol.view(width=800, height=420)
    with open(cfg.protein_pdb) as f:
        view.addModel(f.read(), 'pdb')
    view.setStyle({}, {'cartoon': {'color': COL_PROTEIN, 'opacity': 0.8}})
    view.addModel(mol_to_sdf(reference), 'sdf')
    view.setStyle({'model': 1}, bs(COL_REFERENCE))
    view.addModel(mol_to_sdf(pseudo), 'sdf')
    view.setStyle({'model': 2}, {'sphere': {'radius': 0.23, 'color': color, 'opacity': opacity}})
    view.zoomTo({'model': 2})
    view.rotate(90, 'y')
    view.show()
    print(title)

_pocket_view(pseudo_interior, COL_POCKET, 0.7, 'Full interior (surface_only=False) — all empty voxels in the pocket')
_pocket_view(pseudo_surface,  COL_POCKET, 0.8, 'Surface shell (surface_only=True) — just the 1-voxel-thick boundary')

In [ ]:
def show_pocket_cloud(pseudo, title, color=COL_POCKET, opacity=0.7):
    view = py3Dmol.view(width=800, height=420)
    with open(cfg.protein_pdb) as f:
        view.addModel(f.read(), 'pdb')
    view.setStyle({}, {'cartoon': {'color': COL_PROTEIN, 'opacity': 0.8}})
    view.addSurface(py3Dmol.SAS, {'opacity': 0.9, 'color': 'white'})
    view.addModel(mol_to_sdf(reference), 'sdf')
    view.setStyle({'model': 1}, bs(COL_REFERENCE))
    view.addModel(mol_to_sdf(pseudo), 'sdf')
    view.setStyle({'model': 2}, {'sphere': {'radius': 0.22, 'color': color, 'opacity': opacity}})
    view.zoomTo({'model': 2})
    view.rotate(180, 'y')
    view.rotate(90, 'z')
    view.show()
    print(title)

for label, (pseudo_tmp, ref_score) in results.items():
    show_pocket_cloud(pseudo_tmp, f"{label}  |  ref score: {ref_score:.4f}")

You can see how the full interior fills the whole cavity, the surface-only version is just the shell, and subsampling thins it out further. The scores converge as the representation gets denser, the default `subsample=2` is a good middle ground, as the compute gets to high otherwise.

For the rest of the notebook we use the default settings.

In [ ]:
pseudo, pocket_mask = pocket_to_pseudomol(
    empty, origin, spacing, ref_centroid,
    surface_only=cfg.surface_only,
    subsample=cfg.subsample,
)
ref_score = score_pose(reference, 0, pseudo)
print(f"Pocket atoms: {pseudo.GetNumAtoms()}")
print(f"Reference ligand baseline score: {ref_score:.4f}")

## Step 4 — Conformer Embedding

 We use RDKit's ETKDGv3, to generate conformers, followed by a quick MMFF minimization In this case 100.

In [ ]:
candidate = embed_candidate(cfg.candidate_smiles, n_confs=cfg.n_confs)
print(f"Generated {candidate.GetNumConformers()} conformers for:")
print(f"  {cfg.candidate_smiles}")

In [ ]:
# Show 6 conformers overlaid — gives a sense of how flexible the molecule is
view = py3Dmol.view(width=800, height=400)
colors = ['blue', 'red', 'green', 'orange', 'purple', 'teal']
for i, cid in enumerate(range(min(6, candidate.GetNumConformers()))):
    view.addModel(mol_to_sdf(candidate, conf_id=cid), 'sdf')
    view.setStyle({'model': i}, {'stick': {'color': colors[i], 'opacity': 0.9, 'radius': 0.12}})
view.zoomTo()
view.show()
print("Six random conformers overlaid — each represents a different low-energy shape the molecule can adopt")

## Step 5 — O3A Alignment

Open3DAlign (O3A) overlays the candidate onto the reference ligand by matching atom types and 3D distances, it's a pharmacophore-style superposition. It's fast and gives a physically sensible starting pose, but it only looks at the reference ligand, not the protein pocket.

We run O3A on every conformer and score each against the pocket to find the best O3A-only result. This is our baseline before jitter kicks in.

In [ ]:
o3a_results = []
for cid in range(candidate.GetNumConformers()):
    try:
        align_to_reference(candidate, reference, cid)
        s = score_pose(candidate, cid, pseudo)
        o3a_results.append((s, cid))
    except Exception:
        continue

o3a_results.sort()
best_o3a_score, best_o3a_cid = o3a_results[0]

print(f"Reference baseline:    {ref_score:.4f}")
print(f"Best O3A-only score:   {best_o3a_score:.4f}  (conformer {best_o3a_cid})")
print(f"Gap for jitter to close: {best_o3a_score - ref_score:.4f}")

In [ ]:
view = py3Dmol.view(width=800, height=450)
with open(cfg.protein_pdb) as f:
    view.addModel(f.read(), 'pdb')
view.setStyle({}, {'cartoon': {'color': COL_PROTEIN, 'opacity': 0.7}})
view.addModel(mol_to_sdf(pseudo), 'sdf')
view.setStyle({'model': 1}, {'sphere': {'radius': 0.2, 'color': COL_POCKET, 'opacity': 0.7}})
view.addModel(mol_to_sdf(reference), 'sdf')
view.setStyle({'model': 2}, bs(COL_REFERENCE))
view.addModel(mol_to_sdf(candidate, conf_id=best_o3a_cid), 'sdf')
view.setStyle({'model': 3}, bs(COL_CANDIDATE))
view.zoomTo({'model': 2})
view.show()
print("Orange = reference ligand | Blue = best O3A-aligned candidate | Cyan = pocket surface")

## Step 6 — Jitter

O3A doesn't know about the pocket, it just maximises molecular overlap with the reference. Jitter fixes this (kind of): after each O3A alignment we apply a series of small random rigid-body perturbations (rotation + translation) and keep whatever improves the pocket shape score.

Two parameters:
- `n_jitters`: perturbations per conformer. More = more thorough search.
- The rotation range (±15°) and translation range (±1 Å) are fixed — large enough to see improvements in the score, small enough to stay in the pocket.

Below you can see what jitter looks like: the blue pose is the O3A result, the orange variants are the jittered copies that get evaluated.

In [ ]:
rng_demo = np.random.default_rng(7)
n_demo = 8

view = py3Dmol.view(width=800, height=450)
with open(cfg.protein_pdb) as f:
    view.addModel(f.read(), 'pdb')
view.setStyle({}, {'cartoon': {'color': COL_PROTEIN, 'opacity': 0.5}})
view.addModel(mol_to_sdf(pseudo), 'sdf')
view.setStyle({'model': 1}, {'sphere': {'radius': 0.2, 'color': COL_POCKET, 'opacity': 0.5}})

view.addModel(mol_to_sdf(candidate, conf_id=best_o3a_cid), 'sdf')
view.setStyle({'model': 2}, bs(COL_CANDIDATE))

orig_coords = get_coords(candidate, best_o3a_cid)
for i in range(n_demo):
    angle = rng_demo.uniform(-15, 15, size=3)
    rot = Rotation.from_euler('xyz', angle, degrees=True)
    trans = rng_demo.uniform(-1.0, 1.0, size=3)
    centroid = orig_coords.mean(axis=0)
    jittered = rot.apply(orig_coords - centroid) + centroid + trans
    set_coords(candidate, best_o3a_cid, jittered)
    view.addModel(mol_to_sdf(candidate, conf_id=best_o3a_cid), 'sdf')
    view.setStyle({'model': i + 3}, {'stick': {'color': 'orange', 'opacity': 1.0, 'radius': 0.12}})
set_coords(candidate, best_o3a_cid, orig_coords)

view.zoomTo({'model': 2})
view.show()
print("Blue = O3A pose | Orange = jitter candidates | Cyan = pocket surface")

## Step 7 — Full Aligning Run

Putting it all together: for each conformer → O3A align → score → `n_jitters` random perturbations → keep global best. We report scores at each stage so you can see how much each step contributes.

In [ ]:
best_cid, best_score, best_coords = dock_candidate(
    candidate, reference, pseudo, n_jitters=cfg.n_jitters,
)
set_coords(candidate, best_cid, best_coords)

print(f"Score progression:")
print(f"  Reference baseline:  {ref_score:.4f}")
print(f"  Best O3A-only:       {best_o3a_score:.4f}")
print(f"  After jitter:        {best_score:.4f}")
print(f"  Improvement (jitter vs O3A): {best_o3a_score - best_score:.4f}")
print(f"  Normalised score (vs ref):   {best_score / ref_score:.2f}x baseline")

In [ ]:
view = py3Dmol.view(width=800, height=500)
with open(cfg.protein_pdb) as f:
    view.addModel(f.read(), 'pdb')
view.setStyle({}, {'cartoon': {'color': COL_PROTEIN, 'opacity': 0.8}})
view.addModel(mol_to_sdf(reference), 'sdf')
view.setStyle({'model': 1}, bs(COL_REFERENCE))
view.addModel(mol_to_sdf(candidate, conf_id=best_cid), 'sdf')
view.setStyle({'model': 2}, bs(COL_CANDIDATE))
view.zoomTo({'model': 2})
view.show()
print("Orange = reference ligand | Blue = best aligned pose (after jitter)")

## Step 8 — Torsional Refinement

The docked pose has the right overall position and orientation, but individual torsion angles could be improved to turn atoms into the pseudo molecule. In this refinement step torsion agnles that are connected to atoms that are sticking out of the pseudo mol  are rotated to in `refine_step_deg` increments and greedily accepts angles that reduce protrusion. It focuses only on bonds where at least one side has atoms sticking outside the pocket.

After torsion refinement, an optional constrained MMFF minimization cleans up bond lengths and angles, pocket-buried atoms are pinned in place, only the exposed parts relax.

In [ ]:
score_pre_refine = best_score
e_pre = mmff_energy(candidate, best_cid)

refined_score = refine_pose(
    candidate, best_cid, pseudo, pocket_mask, origin, spacing,
    step_deg=cfg.refine_step_deg, max_iter=cfg.refine_max_iter,
)
e_mid = mmff_energy(candidate, best_cid)

if cfg.minimize_after_refine:
    minimize_pose(
        candidate, best_cid, pocket_mask, origin, spacing,
        core_force=cfg.minimize_core_force,
        max_displacement=cfg.minimize_max_displacement,
    )
    final_score = score_pose(candidate, best_cid, pseudo)
    e_post = mmff_energy(candidate, best_cid)
else:
    final_score = refined_score
    e_post = e_mid

print("Score progression (full pipeline):")
print(f"  Reference baseline:       {ref_score:.4f}")
print(f"  After O3A:                {best_o3a_score:.4f}")
print(f"  After jitter:             {score_pre_refine:.4f}")
print(f"  After torsion refinement: {refined_score:.4f}")
print(f"  After minimization:       {final_score:.4f}")
if e_pre and e_post:
    print(f"\nMMFF energy: {e_pre:.1f} → {e_post:.1f} kcal/mol  (Δ {e_post - e_pre:+.1f})")

In [ ]:
view = py3Dmol.view(width=800, height=500)
with open(cfg.protein_pdb) as f:
    view.addModel(f.read(), 'pdb')
view.setStyle({}, {'cartoon': {'color': COL_PROTEIN, 'opacity': 0.8}})
view.addModel(mol_to_sdf(reference), 'sdf')
view.setStyle({'model': 1}, bs(COL_REFERENCE))
view.addModel(mol_to_sdf(candidate, conf_id=best_cid), 'sdf')
view.setStyle({'model': 2}, bs(COL_CANDIDATE))
view.zoomTo({'model': 2})
view.show()
print("Final refined pose (blue) vs reference ligand (orange)")